In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt

def apply_filter(image, filter_type='sepia'):
    if filter_type == 'sepia':
        kernel = np.array([[0.272, 0.534, 0.131],
                           [0.349, 0.686, 0.168],
                           [0.393, 0.769, 0.189]])
        filtered = cv2.transform(image, kernel)
        filtered = np.clip(filtered, 0, 255)
    elif filter_type == 'gray':
        filtered = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        filtered = cv2.cvtColor(filtered, cv2.COLOR_GRAY2BGR)
    elif filter_type == 'invert':
        filtered = cv2.bitwise_not(image)
    elif filter_type == 'cool':
        cool = cv2.applyColorMap(image, cv2.COLORMAP_COOL)
        filtered = cool
    elif filter_type == 'hot':
        hot = cv2.applyColorMap(image, cv2.COLORMAP_HOT)
        filtered = hot
    elif filter_type == 'smooth_face':
        filtered = smooth_face(image)
    elif filter_type == 'blush':
        filtered = add_blush(image)
    else:
        filtered = image
    return filtered.astype(np.uint8)

def smooth_face(image):
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    result = image.copy()
    for (x, y, w, h) in faces:
        face_roi = result[y:y+h, x:x+w]
        blurred = cv2.bilateralFilter(face_roi, 15, 75, 75)
        result[y:y+h, x:x+w] = blurred
    return result

def add_blush(image):
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    result = image.copy()
    for (x, y, w, h) in faces:
        # Estimate cheek positions
        cheek_radius = int(w * 0.12)
        left_cheek = (x + int(w * 0.32), y + int(h * 0.65))
        right_cheek = (x + int(w * 0.68), y + int(h * 0.65))
        blush_color = (180, 30, 120)  # BGR pinkish
        overlay = result.copy()
        cv2.circle(overlay, left_cheek, cheek_radius, blush_color, -1)
        cv2.circle(overlay, right_cheek, cheek_radius, blush_color, -1)
        alpha = 0.4
        cv2.addWeighted(overlay, alpha, result, 1 - alpha, 0, result)
    return result

def show_camera_with_filter(filter_type='sepia'):
    cap = cv2.VideoCapture(0)
    print("Press 'q' to quit.")
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        filtered_frame = apply_filter(frame, filter_type)
        cv2.imshow('Camera Filter', filtered_frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    cap.release()q
    cv2.destroyAllWindows()

# Example usage:
# show_camera_with_filter('sepia')
# show_camera_with_filter('gray')
# show_camera_with_filter('invert')
# show_camera_with_filter('cool')
# show_camera_with_filter('hot')
show_camera_with_filter('smooth_face')
# show_camera_with_filter('blush')

Press 'q' to quit.


KeyboardInterrupt: 